<a href="https://colab.research.google.com/github/ssindhiyareddy-lab/invoice_bill_ocr_process/blob/main/invoice_bill_code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q easyocr pymupdf opencv-python pillow pandas openpyxl

In [ ]:
import os
import cv2
import re
import fitz
import time
import zipfile
import shutil
import easyocr
import numpy as np
import pandas as pd

from collections import defaultdict
from google.colab import files

In [ ]:
# ===========================================
# STEP 3
# Load EasyOCR
# ===========================================

print("Loading EasyOCR...")

reader = easyocr.Reader(
    ['en'],
    gpu=False
)

print("EasyOCR Loaded Successfully")

In [ ]:
# ===========================================
# STEP 4
# Upload ZIP File
# ===========================================

print("Please upload your ZIP file containing invoices...")

uploaded = files.upload()

# Get uploaded ZIP file name
zip_file = list(uploaded.keys())[0]

print("\nZIP File Uploaded Successfully")
print("ZIP File :", zip_file)

In [ ]:
# ==========================================================
# STEP 5
# Extract ZIP & Find Invoice Files
# ==========================================================

extract_folder = "/content/invoices"

# Delete old folder if it exists
if os.path.exists(extract_folder):
    shutil.rmtree(extract_folder)

# Create new folder
os.makedirs(extract_folder, exist_ok=True)

# Extract ZIP
with zipfile.ZipFile(zip_file, "r") as zip_ref:
    zip_ref.extractall(extract_folder)

print("ZIP Extracted Successfully\n")

# ----------------------------------------------------------
# Find all supported invoice files
# ----------------------------------------------------------

invoice_files = []

supported_formats = (
    ".pdf",
    ".png",
    ".jpg",
    ".jpeg",
    ".tif",
    ".tiff",
    ".bmp"
)

for root, dirs, files in os.walk(extract_folder):

    for file in files:

        if file.lower().endswith(supported_formats):

            invoice_files.append(os.path.join(root, file))

# Sort file names
invoice_files.sort()

print("="*70)
print("TOTAL INVOICE FILES :", len(invoice_files))
print("="*70)

for i, file in enumerate(invoice_files, 1):

    print(f"{i}. {os.path.basename(file)}")

In [ ]:
# ==========================================================
# STEP 6
# Convert PDF/Image into OpenCV Images
# ==========================================================

all_pages = []

print("Loading Invoice Pages...\n")

for file in invoice_files:

    print("Processing :", os.path.basename(file))

    # ----------------------------
    # PDF
    # ----------------------------

    if file.lower().endswith(".pdf"):

        pdf = fitz.open(file)

        print("Pages :", len(pdf))

        for page_no in range(len(pdf)):

            page = pdf.load_page(page_no)

            # Faster rendering (2x instead of 3x)
            pix = page.get_pixmap(matrix=fitz.Matrix(2, 2))

            img = np.frombuffer(
                pix.samples,
                dtype=np.uint8
            )

            img = img.reshape(
                pix.height,
                pix.width,
                pix.n
            )

            if pix.n == 4:
                img = cv2.cvtColor(
                    img,
                    cv2.COLOR_RGBA2BGR
                )
            else:
                img = cv2.cvtColor(
                    img,
                    cv2.COLOR_RGB2BGR
                )

            all_pages.append({

                "file_name": os.path.basename(file),

                "page_no": page_no + 1,

                "image": img

            })

        pdf.close()

    # ----------------------------
    # Images
    # ----------------------------

    else:

        img = cv2.imread(file)

        all_pages.append({

            "file_name": os.path.basename(file),

            "page_no": 1,

            "image": img

        })

print("\n" + "="*70)
print("TOTAL PAGES LOADED :", len(all_pages))
print("="*70)

# Preview loaded pages
for page in all_pages:

    print(
        f"{page['file_name']}  -->  Page {page['page_no']}"
    )

In [ ]:
# ==========================================================
# STEP 7
# OCR + OCR Accuracy
# ==========================================================

from collections import defaultdict
import time

invoice_data = defaultdict(lambda: {
    "rows": [],
    "confidence": [],
    "text": []
})

print("="*70)
print("Starting OCR...")
print("="*70)

start = time.time()

for i, page in enumerate(all_pages):

    print(f"\n[{i+1}/{len(all_pages)}] {page['file_name']} | Page {page['page_no']}")

    image = page["image"]

    # Resize large pages (improves speed)
    h, w = image.shape[:2]

    if w > 1800:

        scale = 1800 / w

        image = cv2.resize(
            image,
            None,
            fx=scale,
            fy=scale,
            interpolation=cv2.INTER_AREA
        )

    result = reader.readtext(
        image,
        detail=1,
        paragraph=False
    )

    for box, text, conf in result:

        invoice_data[page["file_name"]]["rows"].append({

            "page": page["page_no"],

            "bbox": box,

            "text": text.strip(),

            "confidence": float(conf)

        })

        invoice_data[page["file_name"]]["confidence"].append(float(conf))

        invoice_data[page["file_name"]]["text"].append(text.strip())

print("\nOCR Completed")

ocr_results = []

for file_name, data in invoice_data.items():

    if len(data["confidence"]) > 0:

        accuracy = round(
            sum(data["confidence"]) /
            len(data["confidence"])*100,
            2
        )

    else:

        accuracy = 0

    full_text = "\n".join(data["text"])

    ocr_results.append({

        "file_name": file_name,

        "ocr_accuracy": accuracy,

        "rows": data["rows"],

        "text": full_text

    })

end = time.time()

print("\n"+"="*70)

print("Invoices Processed :", len(ocr_results))

print("Total OCR Time :", round(end-start,2),"Seconds")

print("="*70)

In [ ]:
print("\n")

for doc in ocr_results:

    print("="*70)

    print("File Name      :",doc["file_name"])

    print("OCR Accuracy   :",doc["ocr_accuracy"],"%")

    print("OCR Words      :",len(doc["rows"]))

print("="*70)

In [ ]:
print(ocr_results[0].keys())

In [ ]:
print("invoice_results" in globals())

In [ ]:
# ==========================================================
# STEP 8
# Extract Invoice Details
# ==========================================================

import re

invoice_results = []

for doc in ocr_results:

    # Build full OCR text
    text = "\n".join([row["text"] for row in doc["rows"]])

    invoice = {
        "File Name": doc["file_name"],
        "OCR Accuracy": doc["ocr_accuracy"],
        "Invoice Number": "Not Found",
        "Invoice Date": "Not Found",
        "E-Way Bill No": "Not Found"
    }

    # ---------------- Invoice Number ----------------

    patterns = [
        r'Invoice\s*No\.?\s*[:\-]?\s*([A-Za-z0-9\/\-]+)',
        r'Invoice\s*Number\s*[:\-]?\s*([A-Za-z0-9\/\-]+)',
        r'Inv\.?\s*No\.?\s*[:\-]?\s*([A-Za-z0-9\/\-]+)',
        r'Bill\s*No\.?\s*[:\-]?\s*([A-Za-z0-9\/\-]+)',
        r'Tax\s*Invoice\s*No\.?\s*[:\-]?\s*([A-Za-z0-9\/\-]+)'
    ]

    for p in patterns:

        m = re.search(p, text, re.IGNORECASE)

        if m:
            invoice["Invoice Number"] = m.group(1)
            break

    # ---------------- Invoice Date ----------------

    date_patterns = [

        r'Invoice\s*Date\s*[:\-]?\s*([0-9]{2}[/-][0-9]{2}[/-][0-9]{2,4})',

        r'Date\s*[:\-]?\s*([0-9]{2}[/-][0-9]{2}[/-][0-9]{2,4})',

        r'([0-9]{2}[/-][0-9]{2}[/-][0-9]{4})'
    ]

    for p in date_patterns:

        m = re.search(p, text, re.IGNORECASE)

        if m:
            invoice["Invoice Date"] = m.group(1)
            break

    # ---------------- E-Way Bill ----------------

    eway_patterns = [

        r'E[- ]?Way\s*Bill\s*No\.?\s*[:\-]?\s*([0-9]{10,15})',

        r'EWB\s*No\.?\s*[:\-]?\s*([0-9]{10,15})'
    ]

    for p in eway_patterns:

        m = re.search(p, text, re.IGNORECASE)

        if m:
            invoice["E-Way Bill No"] = m.group(1)
            break

    invoice_results.append(invoice)

print("Invoice Details Extracted Successfully")

In [ ]:
for inv in invoice_results:

    print("="*80)
    print("File Name      :", inv["File Name"])
    print("OCR Accuracy   :", inv["OCR Accuracy"])
    print("Invoice Number :", inv["Invoice Number"])
    print("Invoice Date   :", inv["Invoice Date"])
    print("E-Way Bill No  :", inv["E-Way Bill No"])

In [ ]:
# ==========================================================
# STEP 9A
# Detect Table Rows
# ==========================================================

from collections import defaultdict

table_results = []

for doc in ocr_results:

    print("\n" + "="*100)
    print("File :", doc["file_name"])
    print("="*100)

    # Group OCR words into rows using Y-coordinate
    grouped = defaultdict(list)

    for word in doc["rows"]:

        y = int(min(pt[1] for pt in word["bbox"]) / 12)

        grouped[y].append(word)

    table_rows = []

    for key in sorted(grouped.keys()):

        row = sorted(
            grouped[key],
            key=lambda x: min(pt[0] for pt in x["bbox"])
        )

        texts = [w["text"] for w in row]

        table_rows.append(texts)

    # Detect table header
    start = -1

    header_keywords = [
        "description",
        "particular",
        "item",
        "product",
        "goods"
    ]

    for i, row in enumerate(table_rows):

        txt = " ".join(row).lower()

        if any(h in txt for h in header_keywords):

            start = i + 1
            break

    if start == -1:

        print("Table Not Found")

        table_results.append({
            "file_name": doc["file_name"],
            "rows": []
        })

        continue

    extracted_rows = []

    for row in table_rows[start:]:

        line = " ".join(row)

        low = line.lower()

        if any(x in low for x in [
            "grand total",
            "total",
            "cgst",
            "sgst",
            "igst",
            "round off",
            "amount chargeable",
            "terms",
            "bank",
            "authorized"
        ]):
            break

        extracted_rows.append(row)

    table_results.append({

        "file_name": doc["file_name"],

        "rows": extracted_rows

    })

    print("Rows Found :", len(extracted_rows))

In [ ]:
# ==========================================================
# STEP 9B
# Extract Line Items
# ==========================================================

import re

line_items = []

for tbl in table_results:

    print("\n" + "="*100)
    print("File :", tbl["file_name"])
    print("="*100)

    item_no = 1

    invoice_items = []

    for row in tbl["rows"]:

        line = " ".join(row)

        if len(line.strip()) < 5:
            continue

        numbers = re.findall(r"\d[\d,]*\.?\d*", line)

        if len(numbers) < 3:
            continue

        amount = numbers[-1].replace(",", "")

        rate = numbers[-2].replace(",", "")

        qty = numbers[-3].replace(",", "")

        # Unit

        unit = "Not Found"

        m = re.search(
            r"\b(NOS|NO|PCS|PC|UNIT|KG|KGS|LTR|LTRS|BOX|BAG|ROLL|SET|MTR|MT)\b",
            line,
            re.I
        )

        if m:
            unit = m.group().upper()

        # HSN

        hsn = "Not Found"

        m = re.search(r"\b\d{4,8}\b", line)

        if m:
            hsn = m.group()

        # Description

        desc = line

        for n in numbers:
            desc = desc.replace(n, "")

        desc = re.sub(
            r"\b(NOS|NO|PCS|PC|UNIT|KG|KGS|LTR|LTRS|BOX|BAG|ROLL|SET|MTR|MT)\b",
            "",
            desc,
            flags=re.I
        )

        desc = re.sub(r"\|", " ", desc)

        desc = " ".join(desc.split())

        invoice_items.append({

            "Item": item_no,

            "Description": desc,

            "HSN": hsn,

            "Qty": qty,

            "Unit": unit,

            "Rate": rate,

            "Amount": amount

        })

        item_no += 1

    line_items.append({

        "file_name": tbl["file_name"],

        "items": invoice_items

    })

print("Line Item Extraction Completed")

In [ ]:
for tbl in table_results:

    print("\n" + "="*80)
    print(tbl["file_name"])
    print("="*80)

    for row in tbl["rows"]:
        print(" | ".join(row))

In [ ]:
# ==========================================================
# STEP 10
# FINAL REPORT
# ==========================================================

print("\n")
print("="*120)
print("                 MAKER CHECKER OCR REPORT")
print("="*120)

completed_files = []
pending_files = []

# Create lookup for line items
line_item_dict = {}

for item in line_items:
    line_item_dict[item["file_name"]] = item["items"]

# Print invoice-wise report
for invoice in invoice_results:

    file_name = invoice["File Name"]

    print("\n" + "="*120)

    print("File Name      :", file_name)

    print("OCR Accuracy   :", invoice["OCR Accuracy"], "%")

    print("Invoice Number :", invoice["Invoice Number"])

    print("Invoice Date   :", invoice["Invoice Date"])

    print("E-Way Bill No  :", invoice["E-Way Bill No"])

    print("-"*120)

    items = line_item_dict.get(file_name, [])

    if len(items) == 0:

        print("No Line Items Found")

        pending_files.append(file_name)

    else:

        completed_files.append(file_name)

        for itm in items:

            print("\nItem", itm["Item"])
            print("-"*50)

            print("Description :", itm["Description"])
            print("HSN         :", itm["HSN"])
            print("Qty         :", itm["Qty"])
            print("Unit        :", itm["Unit"])
            print("Rate        :", itm["Rate"])
            print("Amount      :", itm["Amount"])

# =====================================================
# Completed Files
# =====================================================

print("\n")
print("="*120)
print("COMPLETED FILES")
print("="*120)

if completed_files:

    for f in completed_files:

        print("✓", f)

else:

    print("None")

# =====================================================
# Pending Files
# =====================================================

print("\n")
print("="*120)
print("PENDING FILES")
print("="*120)

if pending_files:

    for f in pending_files:

        print("✗", f)

else:

    print("None")

# =====================================================
# OCR SUMMARY
# =====================================================

print("\n")
print("="*120)
print("OCR SUMMARY")
print("="*120)

total_accuracy = 0

for invoice in invoice_results:

    print(f"{invoice['File Name']:<60} {invoice['OCR Accuracy']} %")

    total_accuracy += float(invoice["OCR Accuracy"])

if len(invoice_results):

    avg_accuracy = round(total_accuracy / len(invoice_results), 2)

else:

    avg_accuracy = 0

print("\nAverage OCR Accuracy :", avg_accuracy, "%")

print("="*120)
print("PROCESS COMPLETED")
print("="*120)

In [ ]:
import json

final_output = []

for invoice in invoice_results:

    file_name = invoice["File Name"]

    items = []

    for data in line_items:

        if data["file_name"] == file_name:

            items = data["items"]

            break

    final_output.append({

        "file_name": file_name,

        "ocr_accuracy": invoice["OCR Accuracy"],

        "invoice_number": invoice["Invoice Number"],

        "invoice_date": invoice["Invoice Date"],

        "eway_bill_no": invoice["E-Way Bill No"],

        "items": items

    })

print(json.dumps(final_output, indent=4))

In [ ]:
import json

with open("invoice_output.json", "w") as f:

    json.dump(final_output, f, indent=4)

print("JSON file created successfully.")